## Motility induced phase transitions

For particles moving in 2-D,
$$
\frac{d\boldsymbol{x}_i}{dt}
= v_0(\cos\theta_i,\sin\theta_i)-\mu\nabla_i U_{\mathrm{Yukawa}},
\qquad
\frac{d\theta_i}{dt}=\sqrt{2D_r}\,\xi_i(t).
$$

Here $v_0$ is propulsion speed, $\mu$ is mobility, $U_{\mathrm{Yukawa}}$ is the repulsive interaction energy, $D_r$ is rotational diffusion, and $\xi_i(t)$ is independent Gaussian white noise.

The discrete updates are
$$
\boldsymbol{x}_i^{n+1}=\boldsymbol{x}_i^n+
\left[v_0(\cos\theta_i^n,\sin\theta_i^n)-\mu\nabla_i U_{\mathrm{Yukawa}}(\boldsymbol{x}^n)\right]\Delta t,
$$
$$
\theta_i^{n+1}=\theta_i^n+\sqrt{2D_r\Delta t}\,Z_i^n,
\qquad Z_i^n\sim\mathcal N(0,1).
$$
It is important to note that this is overdamped dynamics, as the force is related to the drag!


In [ ]:
import numpy as np


In [ ]:
# Initialization (normalized units; periodic square box).
N = 100
L = 10.0
v_0 = 1.0
mu = 1.0
A = 1.0          # Repulsive Yukawa strength.
kappa = 2.0      # Inverse screening length.
D_r = 0.1       # Rotational diffusion.
dt = 0.001
n_steps = 2000
seed = 42


In [ ]:
# Repulsive pair potential: U(r) = A * exp(-kappa * r) / r.
def yukawa_force(x, L, A, kappa):
    """Total force on each particle, using the nearest periodic image."""
    if A == 0:
        return np.zeros_like(x)
    displacement = x[:, None, :] - x[None, :, :]
    displacement -= L * np.round(displacement / L)
    distance = np.linalg.norm(displacement, axis=2)
    np.fill_diagonal(distance, 1.0)  # Exclude self-interactions below.
    if np.any(distance == 0):
        raise ValueError("Two particles coincide; reduce dt or change initialization.")
    coefficient = A * np.exp(-kappa * distance) * (
        1 / distance**3 + kappa / distance**2)
    np.fill_diagonal(coefficient, 0.0)
    return np.sum(coefficient[:, :, None] * displacement, axis=1)


In [ ]:
# Self-propulsion and rotational diffusion; there is no neighbor alignment.
def propulsion_velocity(theta, v_0):
    return v_0 * np.column_stack((np.cos(theta), np.sin(theta)))


def update_theta(theta, D_r, dt, rng):
    noise = np.sqrt(2 * D_r * dt) * rng.normal(size=len(theta))
    return (theta + noise) % (2 * np.pi)


In [ ]:
# Start on a slightly jittered grid to avoid large initial Yukawa forces.
def initialize_particles(N, L, rng):
    side = int(np.ceil(np.sqrt(N)))
    spacing = L / side
    cells = rng.choice(side**2, size=N, replace=False)
    x = spacing * np.column_stack((cells % side + 0.5, cells // side + 0.5))
    x += rng.uniform(-0.1 * spacing, 0.1 * spacing, size=(N, 2))
    theta = rng.uniform(0, 2 * np.pi, size=N)
    return x % L, theta


In [ ]:
# Overdamped Euler position update and Brownian angle update.
def mips_model(N, L, v_0, mu, A, kappa, D_r, *, dt=0.001,
               n_steps=2000, seed=None):
    """Return x, v, theta, t in a periodic 2-D box.

    x/v: (n_steps+1, N, 2); theta: (n_steps+1, N).
    v is the total instantaneous drift: propulsion + mu * force.
    """
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N must be a positive integer.")
    if not isinstance(n_steps, (int, np.integer)) or n_steps < 0:
        raise ValueError("n_steps must be a nonnegative integer.")
    if not np.all(np.isfinite([L, v_0, mu, A, kappa, D_r, dt])):
        raise ValueError("Parameters must be finite.")
    if L <= 0 or dt <= 0 or min(v_0, mu, A, kappa, D_r) < 0:
        raise ValueError("L/dt must be positive; other parameters must be nonnegative.")

    rng = np.random.default_rng(seed)
    t = np.arange(n_steps + 1) * dt
    x = np.zeros((len(t), N, 2))
    v = np.zeros_like(x)
    theta = np.zeros((len(t), N))
    x[0], theta[0] = initialize_particles(N, L, rng)

    for i in range(len(t)):
        force = yukawa_force(x[i], L, A, kappa)
        v[i] = propulsion_velocity(theta[i], v_0) + mu * force
        if i < n_steps:
            x[i + 1] = (x[i] + dt * v[i]) % L
            theta[i + 1] = update_theta(theta[i], D_r, dt, rng)

    return x, v, theta, t


In [ ]:
x, v, theta, t = mips_model(
    N, L, v_0, mu, A, kappa, D_r,
    dt=dt, n_steps=n_steps, seed=seed
)

# x[i], v[i], theta[i] contain all particles at time t[i].
x.shape, v.shape, theta.shape, t.shape


### Visualization and saving

Arrows/colors show propulsion direction; the chart shows mean total drift speed. The short default run does not guarantee phase separation.

Run the following cells from the `MIPS` folder for inline playback and to save `mips.html` (live controls) and `mips.mp4` (fixed video).


In [ ]:
# Visualization: colors/arrows show propulsion heading; chart shows total drift speed.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, Video, display
from pathlib import Path

mean_speed = np.mean(np.linalg.norm(v, axis=2), axis=1)
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "animation.embed_limit": 80}):
    fig = plt.figure(figsize=(9, 6.5), dpi=85, facecolor="#080e1c")
    ax = fig.add_axes([0.08, 0.13, 0.58, 0.67], facecolor="#0c1629")
    chart = fig.add_axes([0.74, 0.36, 0.21, 0.23], facecolor="#0c1629")
    fig.text(0.08, 0.94, "MIPS / ACTIVE PARTICLES", color="#6ce6ef", fontsize=10, weight="bold")
    fig.text(0.08, 0.875, "Self propulsion meets repulsion.", color="#edf5ff", fontsize=23, weight="bold")
    fig.text(0.08, 0.045, "PERIODIC BOX   /   YUKAWA REPULSION   /   BROWNIAN TURNING", color="#879bb5", fontsize=9)
    ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y")
    ax.set_aspect("equal")
    ax.grid(color="#28405a", alpha=0.35, linewidth=0.5)
    for panel in (ax, chart):
        panel.tick_params(colors="#879bb5", labelsize=8)
        panel.xaxis.label.set_color("#b7c9df")
        panel.yaxis.label.set_color("#b7c9df")
        for spine in panel.spines.values():
            spine.set_color("#28405a")
    arrow_length = 0.025 * L  # Cosmetic length, not a particle diameter or drift magnitude.
    arrows = ax.quiver(x[0, :, 0], x[0, :, 1],
                       arrow_length * np.cos(theta[0]), arrow_length * np.sin(theta[0]),
                       theta[0] % (2 * np.pi), cmap="hsv", clim=(0, 2 * np.pi),
                       angles="xy", scale_units="xy", scale=1, pivot="mid", width=0.005)
    chart.set(xlim=(0, t[-1] if t[-1] > 0 else 1),
              ylim=(0, max(1e-6, float(mean_speed.max())) * 1.05), xlabel="time")
    chart.set_title("MEAN DRIFT SPEED", color="#b7c9df", fontsize=10)
    history, = chart.plot([], [], color="#6ce6ef", lw=1.8)
    fig.text(0.74, 0.22, "Arrows: propulsion heading\nSpeed: propulsion + μ force\nColors show heading", color="#879bb5", fontsize=9, linespacing=1.8, va="top")
    readout = fig.text(0.74, 0.78, "", color="#d3e5f8", family="monospace", va="top")
    fps = 24
    frames = np.unique(np.linspace(0, len(t) - 1, min(len(t), 241)).astype(int))

    def update(i):
        arrows.set_offsets(x[i])
        arrows.set_UVC(arrow_length * np.cos(theta[i]), arrow_length * np.sin(theta[i]),
                       theta[i] % (2 * np.pi))
        history.set_data(t[:i + 1], mean_speed[:i + 1])
        readout.set_text(f"t       {t[i]:6.3f}\nN       {N:6d}\nD_r     {D_r:6.3f}\nspeed   {mean_speed[i]:6.3f}")
        return arrows, history, readout

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / fps,
                               blit=False, repeat=True)
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)

display(HTML(player))


In [ ]:
# Save beside this notebook (run with MIPS as the working directory).
import imageio_ffmpeg
from mips_player import make_mips_player

# Longer, denser HTML preset; the MP4 still uses the Python run above.
controlled_player = make_mips_player(
    N=N, L=7.0, v_0=v_0, mu=mu, A=A, kappa=kappa,
    D_r=0.02, dt=dt, n_steps=100000, seed=seed
)
Path("mips.html").write_text(controlled_player, encoding="utf-8")
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("mips.mp4", writer=FFMpegWriter(fps=fps), dpi=120)

display(Video("mips.mp4", embed=True))


In [ ]:
# KDE average velocity field from the recorded particle motion.
# u(r) = sum K_h(r-x_i(t)) v_i(t) / sum K_h(r-x_i(t)).
# Average vectors first; |u| is different from the average particle speed.
import matplotlib.pyplot as plt


def periodic_kde_velocity(positions, velocities, box_length, bandwidth, *,
                          grid_size=32, min_density=0.0):
    """Return X, Y, mean velocity and number density on a periodic 2-D grid.

    Inputs: (frames, particles, 2), or a single (particles, 2) snapshot.
    Frames have equal weight. Wrapped Gaussian images enforce periodicity;
    image tails beyond at least eight bandwidths are neglected.
    Density is mean particle count per area, not a probability density.
    """
    positions = np.asarray(positions, dtype=float)
    velocities = np.asarray(velocities, dtype=float)
    if positions.ndim == 2:
        positions, velocities = positions[None], velocities[None]
    if (positions.shape != velocities.shape or positions.ndim != 3
            or positions.shape[-1] != 2 or min(positions.shape[:2]) < 1):
        raise ValueError("Use matching nonempty (frames, particles, 2) arrays.")
    if not (np.isfinite(positions).all() and np.isfinite(velocities).all()):
        raise ValueError("Positions and velocities must be finite.")
    if (not np.isfinite([box_length, bandwidth, min_density]).all()
            or box_length <= 0 or bandwidth <= 0 or min_density < 0):
        raise ValueError("L and bandwidth must be positive; density floor nonnegative.")
    if not isinstance(grid_size, (int, np.integer)) or grid_size < 2:
        raise ValueError("grid_size must be an integer of at least 2.")
    coordinates = np.arange(grid_size) * box_length / grid_size
    X, Y = np.meshgrid(coordinates, coordinates)
    queries = np.column_stack((X.ravel(), Y.ravel()))
    weight_sum = np.zeros(len(queries))
    momentum_sum = np.zeros((len(queries), 2))
    images = np.arange(-int(np.ceil(8 * bandwidth / box_length)),
                        int(np.ceil(8 * bandwidth / box_length)) + 1) * box_length
    # Process one frame at a time rather than allocate grid × frames × N.
    for frame_x, frame_v in zip(positions, velocities):
        displacement = queries[:, None, :] - frame_x[None, :, :]
        displacement -= box_length * np.round(displacement / box_length)
        kernel = np.ones(displacement.shape[:2])
        for component in range(2):
            axis_kernel = np.zeros_like(kernel)
            for shift in images:
                axis_kernel += np.exp(-0.5 * ((displacement[..., component] + shift)
                                              / bandwidth)**2)
            kernel *= axis_kernel
        weight_sum += kernel.sum(axis=1)
        momentum_sum += kernel @ frame_v
    density = weight_sum / (len(positions) * 2 * np.pi * bandwidth**2)
    supported = (density >= min_density) & (weight_sum > np.finfo(float).tiny)
    mean_velocity = np.full((len(queries), 2), np.nan)
    mean_velocity[supported] = momentum_sum[supported] / weight_sum[supported, None]
    return X, Y, mean_velocity.reshape(grid_size, grid_size, 2), density.reshape(X.shape)


# Animation controls: zero window gives an instantaneous spatial average.
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, Video, display
import imageio_ffmpeg

kde_bandwidth = 0.06 * L
kde_grid_size = 32
kde_window = 0.0                  # Trailing averaging window in simulation time.
kde_max_window_samples = 7        # Equal-weight snapshots within that window.
kde_density_floor = 0.05 * N / L**2
kde_fps = 24
kde_frames = np.unique(np.linspace(0, len(t) - 1, min(len(t), 121)).astype(int))
kde_fields = []
for kde_index in kde_frames:
    kde_first = np.searchsorted(t, t[kde_index] - kde_window)
    kde_indices = np.unique(np.linspace(kde_first, kde_index,
        min(kde_max_window_samples, kde_index - kde_first + 1)).astype(int))
    kde_X, kde_Y, kde_u, kde_rho = periodic_kde_velocity(
        x[kde_indices], v[kde_indices], L, kde_bandwidth,
        grid_size=kde_grid_size, min_density=kde_density_floor
    )
    kde_fields.append(kde_u)
kde_fields = np.asarray(kde_fields)
kde_speeds = np.linalg.norm(kde_fields, axis=-1)
kde_finite_speeds = kde_speeds[np.isfinite(kde_speeds)]
kde_color_max = max(1e-12, float(kde_finite_speeds.max()) if kde_finite_speeds.size else 0.0)
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "animation.embed_limit": 80}):
    kde_fig, kde_ax = plt.subplots(figsize=(7.2, 6.2), layout="constrained",
                                   facecolor="#080e1c")
    kde_ax.set_facecolor("#0c1629")
    kde_edges = np.linspace(0, L, kde_grid_size + 1)
    kde_background = kde_ax.pcolormesh(
        kde_edges, kde_edges, np.pad(kde_speeds[0], ((0, 1), (0, 1)), mode="wrap"),
        shading="gouraud", cmap="magma", vmin=0, vmax=kde_color_max)
    kde_slice = (slice(None, None, 2),) * 2
    kde_arrows = kde_ax.quiver(kde_X[kde_slice], kde_Y[kde_slice],
        kde_fields[0][kde_slice][..., 0], kde_fields[0][kde_slice][..., 1],
        color="#d7f8ff", angles="xy", scale_units="xy", pivot="mid",
        scale=kde_color_max / (0.045 * L), width=0.004)
    kde_ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y", aspect="equal")
    kde_title = kde_ax.set_title("", color="#edf5ff")
    kde_ax.tick_params(colors="#b7c9df")
    kde_ax.xaxis.label.set_color("#b7c9df")
    kde_ax.yaxis.label.set_color("#b7c9df")
    for spine in kde_ax.spines.values():
        spine.set_color("#28405a")
    kde_bar = kde_fig.colorbar(kde_background, ax=kde_ax, shrink=0.8)
    kde_bar.set_label("Magnitude of KDE mean velocity", color="#b7c9df")
    kde_bar.ax.tick_params(colors="#b7c9df")
    kde_fig.supxlabel("Periodic Gaussian KDE • trailing time window • low-density regions masked",
                     color="#879bb5", fontsize=9)

    def update_kde(frame):
        kde_background.set_array(np.pad(kde_speeds[frame], ((0, 1), (0, 1)), mode="wrap"))
        kde_arrows.set_UVC(kde_fields[frame][kde_slice][..., 0],
                           kde_fields[frame][kde_slice][..., 1])
        kde_title.set_text(f"KDE mean velocity | h={kde_bandwidth:g}\n"
                           f"t={t[kde_frames[frame]]:.3f} | window={kde_window:g}")
        return kde_background, kde_arrows, kde_title

    kde_animation = FuncAnimation(kde_fig, update_kde, frames=len(kde_frames),
                                  interval=1000 / kde_fps, blit=False, repeat=True)
    kde_player = kde_animation.to_jshtml(default_mode="loop")
    plt.close(kde_fig)
display(HTML(kde_player))

# Fixed video of this notebook's KDE run; HTML controls are saved by the earlier export cell.
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    kde_animation.save("mips_kde.mp4", writer=FFMpegWriter(fps=kde_fps), dpi=120)
display(Video("mips_kde.mp4", embed=True))


### MIPS parameter scan: measured diagnostics

The original 95%-uniform-cloud / 50%-of-time classifier was ad hoc; it cannot establish a MIPS phase boundary. Random positional fluctuations, clusters and phase coexistence are different claims. A nonclustered stochastic state is not necessarily chaotic.

Keep the existing **Yukawa active-Brownian model** and scan $\rho=N/L^2$ against $D_r$. This is number density, not disk packing fraction: no fixed particle diameter has been defined. Published soft-disk/WCA phase boundaries do not apply directly to this potential.

Measure two standard observables, averaged over late snapshots and independent seeds:
$$S(k_{\min})=\frac{1}{2N}\left\langle
\left|\sum_j e^{2\pi i x_j/L}\right|^2+
\left|\sum_j e^{2\pi i y_j/L}\right|^2\right\rangle,$$
$$\frac{v_{\parallel}}{v_0}=\frac{1}{Nv_0}\left\langle\sum_j
\boldsymbol v_j\cdot(\cos\theta_j,\sin\theta_j)\right\rangle.$$
Here $k_{\min}=2\pi/L$ is nonzero. Independent uniform positions give $S(k_{\min})=1$; enhanced values indicate large-scale density fluctuations, not proof of coexistence. The second observable measures progress along propulsion heading, rather than speed magnitude; collisions can suppress it. It is undefined when $v_0=0$.

**These defaults are a quick diagnostic scan, not a converged phase diagram.** At $D_r=0.005$, the duration $T=30$ is only $0.15\tau_r$, where $\tau_r=1/D_r$. Hatched cells mark $T D_r<1$; absence of hatching does not establish equilibration. Increase duration and system size, use more seeds, compare early/late measurements and time steps, and inspect density distributions/snapshots before assigning phases. Neither observable alone distinguishes MIPS from every other inhomogeneous state.

The original simulation/animation cells are unchanged. [Active-Brownian equations and model-dependent disk phase diagram](https://arxiv.org/abs/1201.4847); [MIPS diagnostics/background](https://arxiv.org/abs/1406.3533).


In [ ]:
# Independent scan settings; does not change the main simulation or HTML.
phase_N = 64
phase_density = np.array([0.3, 1.0, 3.0])
phase_D_r = np.array([0.005, 0.05, 0.5])
phase_seeds = (42, 43)
phase_dt = 0.001
phase_steps = 30000             # Duration = 30; increase for convergence.
phase_burn_fraction = 0.5       # Discard first half of each run.
phase_samples = 101             # Correlated time samples, not independent trials.
phase_v_0, phase_mu, phase_A, phase_kappa = v_0, mu, A, kappa


In [ ]:
def small_k_structure_factor(positions, box_length):
    """S(k_min), averaged over the two Cartesian wavevectors; (..., N, 2)."""
    positions = np.asarray(positions)
    amplitude = np.mean(np.exp(2j * np.pi * positions / box_length), axis=-2)
    return positions.shape[-2] * np.mean(np.abs(amplitude)**2, axis=-1)


def projected_motility(velocity, headings, propulsion_speed):
    """Mean drift along heading divided by v_0; not mean speed magnitude."""
    if propulsion_speed == 0:
        return np.full(np.asarray(headings).shape[:-1], np.nan)
    direction = np.stack((np.cos(headings), np.sin(headings)), axis=-1)
    return np.mean(np.sum(velocity * direction, axis=-1), axis=-1) / propulsion_speed


In [ ]:
# Sequential runs; preserve main x, v, theta, t and save per-seed observables.
phase_shape = (len(phase_density), len(phase_D_r), len(phase_seeds))
phase_S_min = np.full(phase_shape, np.nan)
phase_motility = np.full(phase_shape, np.nan)
phase_S_early = np.full(phase_shape, np.nan)
phase_final_positions = np.empty(phase_shape + (phase_N, 2))
phase_sample_indices = np.unique(np.linspace(
    int(phase_steps * phase_burn_fraction), phase_steps, phase_samples
).astype(int))
phase_duration = phase_steps * phase_dt
phase_persistence_times = phase_duration * phase_D_r
print("T D_r for the scan:", phase_persistence_times,
      "(values below 1 do not span even one persistence time)", flush=True)
phase_completed = 0
phase_total = int(np.prod(phase_shape))
for row, density in enumerate(phase_density):
    phase_L = np.sqrt(phase_N / density)
    for col, rotational_diffusion in enumerate(phase_D_r):
        for repeat, run_seed in enumerate(phase_seeds):
            phase_x, phase_v, phase_theta, phase_t = mips_model(
                phase_N, phase_L, phase_v_0, phase_mu, phase_A, phase_kappa,
                rotational_diffusion, dt=phase_dt, n_steps=phase_steps, seed=run_seed
            )
            if not (np.isfinite(phase_x).all() and np.isfinite(phase_v).all()):
                raise RuntimeError("Nonfinite scan run: reduce phase_dt.")
            phase_S_min[row, col, repeat] = np.mean(small_k_structure_factor(
                phase_x[phase_sample_indices], phase_L))
            phase_motility[row, col, repeat] = np.mean(projected_motility(
                phase_v[phase_sample_indices], phase_theta[phase_sample_indices], phase_v_0))
            early_indices = np.unique(np.linspace(0, phase_sample_indices[0], phase_samples).astype(int))
            phase_S_early[row, col, repeat] = np.mean(small_k_structure_factor(
                phase_x[early_indices], phase_L))
            phase_final_positions[row, col, repeat] = phase_x[-1]
            del phase_x, phase_v, phase_theta, phase_t
            phase_completed += 1
            print(f"{phase_completed}/{phase_total}: rho={density:g}, "
                  f"D_r={rotational_diffusion:g}, seed={run_seed}, "
                  f"S_min={phase_S_min[row, col, repeat]:.3f}, "
                  f"v_parallel/v_0={phase_motility[row, col, repeat]:.3f}", flush=True)


In [ ]:
# Quantitative diagnostic maps; do not assign phases from arbitrary cutoffs.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path


def logarithmic_edges(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or np.any(values <= 0) or np.any(np.diff(values) <= 0):
        raise ValueError("Use at least two positive, increasing scan values per axis.")
    middle = np.sqrt(values[:-1] * values[1:])
    return np.r_[values[0]**2 / middle[0], middle, values[-1]**2 / middle[-1]]


phase_x_edges = logarithmic_edges(phase_D_r)
phase_y_edges = logarithmic_edges(phase_density)
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10}):
    phase_fig, phase_axes = plt.subplots(1, 2, figsize=(11, 5.4), layout="constrained",
                                         facecolor="#080e1c")
    phase_fig.suptitle("Yukawa ABP / finite-time parameter scan", color="#edf5ff", fontsize=18)
    observables = [phase_S_min.mean(axis=2), phase_motility.mean(axis=2)]
    titles = [r"Density fluctuations: $S(k_{\min})$", r"Projected motility: $v_{\parallel}/v_0$"]
    for panel, values, title in zip(phase_axes, observables, titles):
        mesh = panel.pcolormesh(phase_x_edges, phase_y_edges, values,
                               cmap="viridis", shading="flat")
        panel.set_title(title, color="#edf5ff")
        panel.set_facecolor("#0c1629")
        panel.set(xscale="log", yscale="log", xlabel=r"Rotational diffusion $D_r$",
                  ylabel=r"Number density $\rho=N/L^2$")
        panel.set_xticks(phase_D_r, [f"{value:g}" for value in phase_D_r])
        panel.set_yticks(phase_density, [f"{value:g}" for value in phase_density])
        panel.minorticks_off()
        panel.tick_params(colors="#b7c9df")
        panel.xaxis.label.set_color("#b7c9df")
        panel.yaxis.label.set_color("#b7c9df")
        for spine in panel.spines.values():
            spine.set_color("#28405a")
        for row in range(len(phase_density)):
            for col in range(len(phase_D_r)):
                if phase_persistence_times[col] < 1:
                    panel.add_patch(Rectangle(
                        (phase_x_edges[col], phase_y_edges[row]),
                        phase_x_edges[col+1]-phase_x_edges[col],
                        phase_y_edges[row+1]-phase_y_edges[row],
                        fill=False, edgecolor="#d6dce6", hatch="//", linewidth=0))
                panel.text(phase_D_r[col], phase_density[row], f"{values[row, col]:.2f}",
                           ha="center", va="center", color="white", fontsize=9,
                           bbox=dict(facecolor="#080e1c", alpha=0.65, edgecolor="none", pad=2))
        bar = phase_fig.colorbar(mesh, ax=panel, shrink=0.8)
        bar.ax.tick_params(colors="#b7c9df")
    phase_fig.supxlabel(
        f"N={phase_N} | T={phase_duration:g} | {len(phase_seeds)} seeds | "
        "Hatching: T Dᵣ < 1 | Diagnostics, not a confirmed phase boundary",
        color="#879bb5", fontsize=9)
    phase_fig.savefig("mips_phase_map.png", dpi=180, facecolor=phase_fig.get_facecolor())
    plt.show()

np.savez_compressed(
    "mips_phase_scan.npz", diagnostic_version=2, density=phase_density, D_r=phase_D_r,
    S_min=phase_S_min, S_early=phase_S_early, projected_motility=phase_motility,
    final_positions=phase_final_positions, persistence_times=phase_persistence_times,
    N=phase_N, seeds=phase_seeds, dt=phase_dt, n_steps=phase_steps,
    sample_indices=phase_sample_indices, burn_fraction=phase_burn_fraction,
    v_0=phase_v_0, mu=phase_mu, A=phase_A, kappa=phase_kappa
)
print("No phase labels assigned: check duration, system-size and time-step convergence.")
print("Saved mips_phase_map.png and mips_phase_scan.npz in", Path.cwd())
